In [ ]:
# ─────────────────────────────────────────────────────────────
#Randomly selects N CSV files from a directory of 63 CSV files and stacks 
# them into one combined output file.
# ─────────────────────────────────────────────────────────────


import os
import random
import pandas as pd

# ─────────────────────────────────────────────────────────────
# CONFIGURATION  ← edit these
# ─────────────────────────────────────────────────────────────

ROOT_DIR     = "."  
N            = 15                           
LABEL_COL    = "Label"                     
OUTPUT_FILE  = "combined_output.csv"       
RANDOM_SEED  = 42                          

# They guarantee Spoofing + Web_Attack are present
FORCE_INCLUDE = [
    "Merged27.csv",  
    "Merged08.csv",  
]

# ─────────────────────────────────────────────────────────────
# LABEL MAPPING — uppercase matching covers all variants
# ─────────────────────────────────────────────────────────────
REQUIRED_CATEGORIES = {
    "Benign":         lambda l: l == "BENIGN",
    "DDoS":           lambda l: l.startswith("DDOS-"),
    "DoS":            lambda l: l.startswith("DOS-"),
    "Mirai":          lambda l: l.startswith("MIRAI-") or l == "BACKDOOR_MALWARE",
    "Reconnaissance": lambda l: l.startswith("RECON-") or l == "VULNERABILITYSCAN",
    "Spoofing":       lambda l: l in ["DNS_SPOOFING", "MITM-ARPSPOOFING"],
    "Brute_Force":    lambda l: l == "DICTIONARYBRUTEFORCE",
    "Web_Attack":     lambda l: l in ["SQLINJECTION", "XSS", "COMMANDINJECTION",
                                       "UPLOADING_ATTACK", "BROWSERHIJACKING"],
}

# ─────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────

def get_csv_files(directory):
    files = [
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if f.lower().endswith(".csv")
    ]
    if not files:
        raise FileNotFoundError(f"No CSV files found in: {directory}")
    return files


def file_has_category(filepath, label_col, check_fn):
    try:
        df = pd.read_csv(filepath, usecols=[label_col])
        labels = (df[label_col]
                  .dropna()                  
                  .astype(str)
                  .str.upper()
                  .str.strip())
        return labels.apply(check_fn).any()
    except Exception as e:
        print(f"  Could not read {os.path.basename(filepath)}: {e}")
        return False


def scan_files_for_categories(all_files, label_col, required_categories):
    print(f"\nScanning {len(all_files)} files for required categories...")
    coverage = {cat: [] for cat in required_categories}
    for i, path in enumerate(all_files, start=1):
        print(f"  [{i}/{len(all_files)}] Scanning {os.path.basename(path)}...", end="\r")
        for cat, check_fn in required_categories.items():
            if file_has_category(path, label_col, check_fn):
                coverage[cat].append(path)
    print("\n")
    return coverage


def select_files(all_files, n, coverage, force_include=None, seed=None):

    random.seed(seed)
    guaranteed = set()

    # Step 1: Force include
    if force_include:
        print("Forced inclusions:")
        for fname in force_include:
            matches = [f for f in all_files if os.path.basename(f) == fname]
            if matches:
                guaranteed.add(matches[0])
                print(f" included: {fname}")
            else:
                print(f" Force include '{fname}' not found in directory — skipping")

    # Step 2: Guarantee category coverage
    print("\nCategory coverage:")
    for cat, files in coverage.items():
        if not files:
            print(f" '{cat}': no file found in entire directory — cannot include")
            continue

        # Check if any already-guaranteed file covers this category
        already_covered = any(
            file_has_category(f, LABEL_COL, REQUIRED_CATEGORIES[cat])
            for f in guaranteed
        )

        if already_covered:
            print(f" '{cat}': already covered by a forced file")
        else:
            chosen = random.choice(files)
            guaranteed.add(chosen)
            print(f" '{cat}': covered by {os.path.basename(chosen)}")

    # Warn if coverage requires more files than N
    if len(guaranteed) > n:
        print(f"\nWarning:  Coverage requires {len(guaranteed)} files but N={n}. "
              f"Increasing selection size to {len(guaranteed)}.")
        return list(guaranteed)

    # Step 3: Random fill
    remaining_pool = [f for f in all_files if f not in guaranteed]
    random.shuffle(remaining_pool)
    fill = remaining_pool[:n - len(guaranteed)]
    selected = list(guaranteed) + fill
    random.shuffle(selected)
    return selected


def stack_csvs(file_paths):
    print(f"\nStacking {len(file_paths)} CSV files...")
    dfs = []
    for i, path in enumerate(file_paths, start=1):
        df = pd.read_csv(path)
        print(f"  [{i}] {os.path.basename(path)}  → {df.shape[0]:,} rows, {df.shape[1]} cols")
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────

def main():
    all_files = get_csv_files(ROOT_DIR)
    print(f"Found {len(all_files)} CSV files in '{ROOT_DIR}'")

    coverage = scan_files_for_categories(all_files, LABEL_COL, REQUIRED_CATEGORIES)

    print("Category coverage summary:")
    for cat, files in coverage.items():
        print(f"  {cat:20s} → found in {len(files)} file(s)")

    print(f"\nSelecting {N} files...")
    selected_files = select_files(
        all_files, N, coverage,
        force_include=FORCE_INCLUDE,
        seed=RANDOM_SEED
    )

    print(f"\nFinal selection ({len(selected_files)} files):")
    for f in selected_files:
        print(f"  • {os.path.basename(f)}")

    combined = stack_csvs(selected_files)

    if LABEL_COL in combined.columns:
        print("\nLabel distribution in combined dataset:")
        print(combined[LABEL_COL].value_counts().to_string())

    # Step 6: Save
    combined.to_csv(OUTPUT_FILE, index=False)
    print(f"\nOK: Done! Saved to: '{OUTPUT_FILE}'")
    print(f"   Final shape: {combined.shape[0]:,} rows × {combined.shape[1]} columns")


if __name__ == "__main__":
    main()